# Real Data Results — *E. coli* K-12 vs O157:H7
Comparing **V1 (naive DP)** vs **V2 (banded merge DP)** on 19,124 × 21,372 GATC methylation sites.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
COLORS = {'V1': '#4878CF', 'V2': '#E84855'}
RES    = '../results/real'
EPS    = 50

# ── summaries ──────────────────────────────────────────────────────────────
with open(f'{RES}/v1/summary.json') as f: s1 = json.load(f)
with open(f'{RES}/v2/summary.json') as f: s2 = json.load(f)

# ── anchor coords ──────────────────────────────────────────────────────────
anc1 = pd.read_csv(f'{RES}/v1/anchor_coords.tsv', sep='\t')
anc2 = pd.read_csv(f'{RES}/v2/anchor_coords.tsv', sep='\t')

# ── traces ─────────────────────────────────────────────────────────────────
tr1 = pd.read_csv(f'{RES}/v1/alignment_trace.tsv', sep='\t')
tr2 = pd.read_csv(f'{RES}/v2/alignment_trace.tsv', sep='\t')

# ── gap positions (V1) ─────────────────────────────────────────────────────
gp1 = pd.read_csv(f'{RES}/v1/gap_positions.tsv', sep='\t')

print('V1 summary:', s1)
print('V2 summary:', s2)

V1 summary: {'n_A_sites': 19124, 'n_B_sites': 21372, 'n_matched': 17374, 'n_gap_a': 3997, 'n_gap_b': 1749, 'match_rate_A': 0.9085, 'match_rate_B': 0.8129, 'runtime_s': 553.41, 'error_mean_bp': 14.0, 'error_median_bp': 0.0, 'error_std_bp': 37.11, 'error_p90_bp': 52.0, 'error_p99_bp': 186.0, 'frac_within_eps': 0.8977, 'frac_exact_match': 0.7535, 'best_score': 1034529.0}
V2 summary: {'n_A_sites': 19124, 'n_B_sites': 21372, 'n_matched': 16651, 'n_gap_a': 1743, 'n_gap_b': 392, 'match_rate_A': 0.8707, 'match_rate_B': 0.7791, 'runtime_s': 8497.62, 'error_mean_bp': 3.01, 'error_median_bp': 0.0, 'error_std_bp': 8.93, 'error_p90_bp': 11.0, 'error_p99_bp': 44.0, 'frac_within_eps': 1.0, 'frac_exact_match': 0.8354, 'best_score': 1408447.0, 'n_merge': 2563}


## 1 · Key Metrics Comparison

In [12]:
def fmt_val(v, is_rate):
    """Format a metric value: rates as percentages, bp values as floats."""
    if v is None:
        return 'N/A'
    if is_rate:
        return f'{v * 100:.1f}%'
    return f'{v:,.2f}'

def better_marker(v1, v2, lower_is_better):
    """Return ▲ next to the better value."""
    if v1 == v2:
        return '', ''
    if lower_is_better:
        return ('▲', '') if v1 < v2 else ('', '▲')
    else:
        return ('▲', '') if v1 > v2 else ('', '▲')

# (key, display label, is_rate, lower_is_better)
REPORT_METRICS = [
    ('match_rate_A',     'Match Rate (A sites)',       True,  False),
    ('frac_within_eps',  f'Frac within eps={EPS} bp',  True,  False),
    ('frac_exact_match', 'Exact Match Fraction',       True,  False),
    ('error_mean_bp',    'Mean Interval Error (bp)',   False, True),
    ('error_median_bp',  'Median Interval Error (bp)', False, True),
    ('error_p90_bp',     'P90 Error (bp)',              False, True),
    ('error_p99_bp',     'P99 Error (bp)',              False, True),
    ('runtime_s',        'Runtime (s)',                 False, True),
]

# ── build rows ────────────────────────────────────────────────────────────────
rows = []
for key, label, is_rate, lower_is_better in REPORT_METRICS:
    v1, v2 = s1[key], s2[key]
    m1, m2 = better_marker(v1, v2, lower_is_better)
    rows.append({
        'Metric':  label,
        'V1':      f'{fmt_val(v1, is_rate)}{" " + m1 if m1 else ""}',
        'V2':      f'{fmt_val(v2, is_rate)}{" " + m2 if m2 else ""}',
    })

# ── gap / move counts ─────────────────────────────────────────────────────────
expected_net = s1['n_B_sites'] - s1['n_A_sites']
for key, label in [('n_matched',  'Matched pairs'),
                   ('n_gap_a',    'Gaps in A (skipped B sites)'),
                   ('n_gap_b',    'Gaps in B (skipped A sites)')]:
    v1, v2 = s1[key], s2[key]
    rows.append({'Metric': label, 'V1': f'{v1:,}', 'V2': f'{v2:,}'})

rows.append({'Metric': 'Net gaps  (gap_A − gap_B)',
             'V1': f'{s1["n_gap_a"] - s1["n_gap_b"]:,}',
             'V2': f'{s2["n_gap_a"] - s2["n_gap_b"]:,}'})
rows.append({'Metric': 'Expected net  (|B| − |A|)',
             'V1': f'{expected_net:,}',
             'V2': f'{expected_net:,}'})
if 'n_merge' in s2:
    rows.append({'Metric': 'Merge moves (V2 only)', 'V1': '—', 'V2': f'{s2["n_merge"]:,}'})

# ── render as styled DataFrame ────────────────────────────────────────────────
tbl = pd.DataFrame(rows).set_index('Metric')

display(
    tbl.style
    .set_caption('E. coli K-12 vs O157:H7 — Alignment Quality Metrics')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '15px'), ('font-weight', 'bold'),
                   ('text-align', 'left'), ('padding-bottom', '10px'),
                   ('white-space', 'nowrap')]},
        {'selector': 'th',
         'props': [('background-color', '#2c3e50'), ('color', 'white'),
                   ('font-size', '13px'), ('padding', '7px 16px'),
                   ('text-align', 'center')]},
        {'selector': 'td',
         'props': [('font-size', '13px'), ('padding', '6px 16px'),
                   ('border-bottom', '1px solid #ddd'), ('text-align', 'right')]},
        {'selector': 'th:first-child, td:first-child',
         'props': [('text-align', 'left')]},
        {'selector': 'tr:hover td',
         'props': [('background-color', '#f0f4f8')]},
    ])
    .apply(lambda col: [
        'background-color: #eaf4fb' if col.name == 'V1' else
        'background-color: #fef9f0' if col.name == 'V2' else ''
        for _ in col
    ])
)
print('▲ = better value')


,V1,V2
Metric,,
Match Rate (A sites),90.8% ▲,87.1%
Frac within eps=50 bp,89.8%,100.0% ▲
Exact Match Fraction,75.3%,83.5% ▲
Mean Interval Error (bp),14.00,3.01 ▲
Median Interval Error (bp),0.00,0.00
P90 Error (bp),52.00,11.00 ▲
P99 Error (bp),186.00,44.00 ▲
Runtime (s),553.41 ▲,"8,497.62"
Matched pairs,"17,374","16,651"


▲ = better value


In [17]:
tbl.style.to_excel("../results/tables/alignment_metrics.xlsx", engine="openpyxl")